# 0. 공통: 오늘의 브리핑, 가게 진단, 챗봇 연결

판매자용 챗봇 노트북 모음의 첫 번째예요.

| 노트북 | 메뉴 |
| --- | --- |
| **0_공통** | 🏠 오늘의 브리핑, 가게 진단, 자유 질문, 챗봇에 연결하는 법 |
| 1_배송지연_쿠폰 | 지연 책임, 발송 what-if, 도착일 안내, 쿠폰 금액·예산, 약속일 임박 경고 |
| 2_평점_리뷰 | 별점·불만 유형, 리뷰 번역·답글, 1~2점 위험 주문, 나쁜 리뷰의 값, 묶음 주문 |
| 3_내고객이해 | 약속 vs 기간, 지역 불만(쓰나미식 비율 barh), 권역별 배송일, 리뷰 속도, 날짜 세는 고객, 도착 전 리뷰, 지역 기회 |
| 4_발주 | 수요 예측, 발주량, 품절 위험 조정, 발주 우선순위, 이벤트 배수 |
| 5_생산량_매출 | 시기별 매출 증감, 카테고리 순위·파이, 가격·운송비, 생산 계획 |
| 6_블랙프라이데이 | 작년 BF 성적표, 카테고리별 배수, 내 가게 BF, 준비 체크리스트, 할인 시뮬레이터, 다른 이벤트 |

**실행법 (Colab)**: GitHub 저장소(jc020230/BDA)에서 이 노트북을 Colab으로 열고 위에서부터 실행하면 돼요. 첫 셀이 저장소를 자동으로 내려받아서 `seller_tools.py`와 `preprocessing/` 폴더의 데이터를 찾아요. 노트북끼리는 서로 독립이라 아무거나 먼저 열어도 돼요.

**아래 "챗봇이 이렇게 말할 수 있어요"** 문장은 도구 결과 숫자로 자동으로 만든 예시예요. 실제 챗봇에서는 LLM이 이 결과를 받아 자연스럽게 말해요.

In [1]:
# 준비: 저장소 파일(seller_tools.py, 전처리 데이터)을 찾고, 없으면 GitHub에서 내려받기
import os, sys, json, zipfile, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jc020230/BDA.git'
DATA_DIRS = ['olist_mart', 'preprocessing', '/content/drive/MyDrive/olist_mart']
has_data = lambda p: os.path.exists(f'{p}/orders_mart.parquet')

# Colab에서 GitHub 노트북만 열면 다른 파일이 없어서, 저장소를 통째로 내려받아 그 폴더로 이동
if not os.path.exists('seller_tools.py'):
    if not os.path.exists('BDA/seller_tools.py'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'BDA'], check=True)
    os.chdir('BDA')
MART_DIR = next((p for p in DATA_DIRS if has_data(p)), None)
if MART_DIR is None:   # 저장소에 데이터가 없으면 zip이나 드라이브에서 찾기
    z = next((p for p in ['olist_mart_v2_github.zip', 'olist_mart_v2.zip', 'olist_mart.zip'] if os.path.exists(p)), None)
    if z: zipfile.ZipFile(z).extractall('.'); MART_DIR = 'olist_mart'
    elif IN_COLAB:
        from google.colab import drive; drive.mount('/content/drive'); MART_DIR = next(p for p in DATA_DIRS if has_data(p))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from seller_tools import SellerTools, TOOLS, setup_korean_font
setup_korean_font()
st = SellerTools(MART_DIR)
if st.evm_table is None:
    print('⚠️ 전처리 1판 결과예요. 2판을 쓰면 이벤트 배수 표와 도착 전 리뷰 기능까지 쓸 수 있어요.')

def show(result, limit=1500):
    """도구 결과(JSON)를 보기 좋게 출력"""
    text = json.dumps(result, ensure_ascii=False, indent=1)
    print(text if len(text) <= limit else text[:limit] + '\n... (생략)')

DEMO = '2eb70248'                  # 시계·선물, 별점 낮고 발송이 느린 판매자 (앞 8자리만 써도 됨)
BIG = st.ss.orders.idxmax()        # 주문이 가장 많은 판매자
print('도구', len(TOOLS), '개 준비 완료 | 작업 폴더:', os.getcwd(), '| 데이터:', MART_DIR)

도구 23 개 준비 완료 | 작업 폴더: /tmp/repo_asof | 데이터: preprocessing


## 🏠 오늘의 브리핑
판매자가 챗봇을 열면 먼저 보여줄 요약이에요. 데이터가 과거 것이라 "오늘"은 그 판매자의 마지막 주문일로 잡아요 (`as_of`로 바꿀 수 있어요).

In [2]:
r = st.call('store_briefing', {'seller_id': DEMO})
show(r)
ev = r['upcoming_events'][0] if r['upcoming_events'] else None
print('\n💬 챗봇이 이렇게 말할 수 있어요:')
print(f"최근 30일 주문 {r['orders']}건, 지연 {r['late_orders']}건(그중 발송 지연 {r['late_due_to_seller_shipping']}건), 새 1~2점 리뷰 {r['new_low_reviews']}건이에요. "
      f"같은 카테고리 판매자 {r['peer_n']}명 중 별점 {int(r['score_rank_in_peer'])}위예요. 지금 가장 먼저 할 일은 '{r['top_action']['priority']}'이에요."
      + (f" {ev['days_left']}일 뒤 {ev['event']}가 있어요 (지난번 시장 {ev['market_multiple']:.2f}배)." if ev else ''))

{
 "seller_id": "2eb70248d66e0e3ef83659f71b244378",
 "period": [
  "2018-06-19",
  "2018-07-19"
 ],
 "orders": 59,
 "revenue": 13036.28,
 "late_orders": 1,
 "late_due_to_seller_shipping": 1,
 "new_low_reviews": 10,
 "new_reviews": 23,
 "score_rank_in_peer": 23.0,
 "peer_n": 23,
 "top_action": {
  "priority": "발송 마감 지키기",
  "why": "마감 위반 32% (동종 4%), 지연 도착 21건 중 18건이 발송 지연 때문"
 },
 "upcoming_events": [
  {
   "event": "아버지날",
   "date": "2018-08-12",
   "days_left": 16,
   "market_multiple": 1.1227
  }
 ]
}

💬 챗봇이 이렇게 말할 수 있어요:
최근 30일 주문 59건, 지연 1건(그중 발송 지연 1건), 새 1~2점 리뷰 10건이에요. 같은 카테고리 판매자 23명 중 별점 23위예요. 지금 가장 먼저 할 일은 '발송 마감 지키기'이에요. 16일 뒤 아버지날가 있어요 (지난번 시장 1.12배).


## 가게 종합 진단
같은 대표 카테고리 + 주문 30건 이상 판매자를 "동종"으로 보고 비교한 뒤, **판매자가 바꿀 수 있는 것** 중에서 무엇부터 고칠지 정해요.

In [3]:
r = st.call('seller_diagnosis', {'seller_id': DEMO})
show(r, 2500)

{
 "seller_id": "2eb70248d66e0e3ef83659f71b244378",
 "state": "SP",
 "city": "campinas",
 "main_category": "시계·선물",
 "orders": 187,
 "revenue": 38990.72,
 "reliable": true,
 "me": {
  "avg_score": 2.7845,
  "low_rate": 0.4751,
  "late_rate": 0.1123,
  "handling_days_median": 9.604,
  "ship_late_rate": 0.3209
 },
 "peer_median": {
  "avg_score": 4.25,
  "low_rate": 0.1034,
  "late_rate": 0.0541,
  "handling_days_median": 1.8641,
  "ship_late_rate": 0.0395
 },
 "peer_n": 23,
 "score_rank_in_peer": 23.0,
 "delay_breakdown": {
  "orders": 187,
  "late_orders": 21,
  "late_rate": 0.1123,
  "late_due_to_seller_shipping": 18,
  "late_in_transit_only": 3,
  "late_rate_if_shipped_late": 0.3,
  "late_rate_if_shipped_on_time": 0.0236
 },
 "actions": [
  {
   "priority": "발송 마감 지키기",
   "why": "마감 위반 32% (동종 4%), 지연 도착 21건 중 18건이 발송 지연 때문"
  },
  {
   "priority": "상품·검수 점검",
   "why": "제때 보낸 주문도 47%가 1~2점 (동종 10%), 가장 많은 불만은 다른 상품 20%"
  },
  {
   "priority": "묶음 주문 관리",
   "why": "상품 2개 이상 주문의 1~

In [4]:
# 같은 카테고리의 대형 판매자는 진단이 완전히 달라요
r = st.call('seller_diagnosis', {'seller_id': BIG})
show({'me': r['me'], 'peer_median': r['peer_median'], 'delay_breakdown': r['delay_breakdown'], 'actions': r['actions']})

{
 "me": {
  "avg_score": 3.847,
  "low_rate": 0.1866,
  "late_rate": 0.098,
  "handling_days_median": 1.8862,
  "ship_late_rate": 0.0887
 },
 "peer_median": {
  "avg_score": 3.997,
  "low_rate": 0.1734,
  "late_rate": 0.0661,
  "handling_days_median": 1.9258,
  "ship_late_rate": 0.0504
 },
 "delay_breakdown": {
  "orders": 1724,
  "late_orders": 169,
  "late_rate": 0.098,
  "late_due_to_seller_shipping": 39,
  "late_in_transit_only": 130,
  "late_rate_if_shipped_late": 0.2549,
  "late_rate_if_shipped_on_time": 0.0827
 },
 "actions": [
  {
   "priority": "현재 수준 유지",
   "why": "동종 대비 뚜렷한 약점이 없어요"
  }
 ]
}


## 자유 질문
메뉴에 없는 질문은 LLM이 pandas 코드를 짜서 답할 수 있어요. 이때 **그 판매자 데이터만 거른 표**를 주면 다른 판매자 정보가 섞이지 않아요. 열 설명은 `st.DATA_DICTIONARY`에 있어요.

In [5]:
my = st.seller_data(DEMO)
print({k: v.shape for k, v in my.items()})
# 예: "작년 11월 내 매출은?" → LLM이 이런 코드를 짜게 하면 돼요
o = my['orders']
print('2017년 11월 매출:', round(o[(o.year_month == '2017-11') & o.is_sale].price.sum(), 2), '헤알')

{'orders': (202, 48), 'items': (212, 25), 'reviews': (193, 32), 'weekly': (250, 8), 'profile': (1, 24)}
2017년 11월 매출: 0.0 헤알


## 이미 있는 챗봇에 연결하기

1. **도구 설명서를 LLM에 넘기기**: `st.tools_for('openai' | 'anthropic' | 'gemini')`
2. **LLM이 고른 도구를 실행하기**: `result = st.call(이름, 입력값)` → JSON으로 바꿔 LLM에 돌려주기. 에러가 나도 `{'error': ...}`가 와서 멈추지 않아요.
3. **그래프**: `fig = st.plot('plot_이름', ...)` → 이미지로 저장해서 챗봇 화면에 붙이기

In [6]:
print([t['name'] for t in TOOLS])
print('그래프:', ['plot_region_days', 'plot_delay_vs_score', 'plot_state_complaints', 'plot_sales_trend', 'plot_category_pie', 'plot_category_rank', 'plot_forecast'])
show(st.tools_for('anthropic')[0])

['store_briefing', 'seller_diagnosis', 'delay_breakdown', 'shipping_whatif', 'delivery_estimate', 'coupon_advice', 'coupon_budget', 'promise_alerts', 'review_summary', 'review_examples', 'low_review_risk', 'review_cost', 'customer_insight', 'demand_forecast', 'reorder_quantity', 'reorder_priority', 'production_plan', 'sales_trend', 'category_ranking', 'price_position', 'event_report', 'discount_simulator', 'event_checklist']
그래프: ['plot_region_days', 'plot_delay_vs_score', 'plot_state_complaints', 'plot_sales_trend', 'plot_category_pie', 'plot_category_rank', 'plot_forecast']
{
 "name": "store_briefing",
 "description": "[홈] 오늘의 가게 브리핑: 최근 30일 주문·지연·나쁜 리뷰, 동종 순위, 1순위 할 일, 다가오는 이벤트",
 "input_schema": {
  "type": "object",
  "properties": {
   "seller_id": {
    "type": "string",
    "description": "판매자 ID (앞 8자리만 줘도 됨)"
   },
   "as_of": {
    "type": "string",
    "description": "기준 날짜. 비우면 판매자의 마지막 주문일"
   }
  },
  "required": [
   "seller_id"
  ]
 }
}


In [7]:
SYSTEM_PROMPT = """너는 브라질 전자상거래 플랫폼 Olist의 판매자를 돕는 한국어 비서야.
- 지금 대화 중인 판매자 ID는 {seller_id}야. 도구를 부를 때 이 ID를 써.
- 숫자(별점, 비율, 금액, 예측값)는 반드시 도구 결과에 있는 것만 말해. 없는 숫자는 지어내지 마.
- 비율은 퍼센트로, 금액은 헤알(R$)로 말해.
- 도구 결과에 note, confidence, basis가 있으면 한계도 짧게 같이 말해 (예: 쿠폰 효과는 가정값, 예측 신뢰도 낮음).
- 리뷰 원문은 포르투갈어야. 요청하면 한국어로 번역·요약하고, 답글 초안은 포르투갈어로 쓰고 한국어 뜻을 같이 보여줘.
- 메뉴: 🏠 브리핑·진단 / 1 배송·쿠폰 / 2 평점·리뷰 / 3 내 고객 이해 / 4 발주 / 5 생산량·매출 / 6 블랙프라이데이·이벤트
"""
print(SYSTEM_PROMPT.format(seller_id=DEMO))

너는 브라질 전자상거래 플랫폼 Olist의 판매자를 돕는 한국어 비서야.
- 지금 대화 중인 판매자 ID는 2eb70248야. 도구를 부를 때 이 ID를 써.
- 숫자(별점, 비율, 금액, 예측값)는 반드시 도구 결과에 있는 것만 말해. 없는 숫자는 지어내지 마.
- 비율은 퍼센트로, 금액은 헤알(R$)로 말해.
- 도구 결과에 note, confidence, basis가 있으면 한계도 짧게 같이 말해 (예: 쿠폰 효과는 가정값, 예측 신뢰도 낮음).
- 리뷰 원문은 포르투갈어야. 요청하면 한국어로 번역·요약하고, 답글 초안은 포르투갈어로 쓰고 한국어 뜻을 같이 보여줘.
- 메뉴: 🏠 브리핑·진단 / 1 배송·쿠폰 / 2 평점·리뷰 / 3 내 고객 이해 / 4 발주 / 5 생산량·매출 / 6 블랙프라이데이·이벤트



In [8]:
# 예측 실험 결과(predictor_experiments.ipynb)를 돌렸다면, 챗봇이 예측 오차를 같이 말할 때 이 파일을 쓰면 돼요
p = f'{MART_DIR}/experiments/model_report.json'
print(json.dumps(json.load(open(p, encoding='utf-8'))['summary'], ensure_ascii=False, indent=1) if os.path.exists(p) else '아직 실험 결과가 없어요 (predictor_experiments.ipynb 실행 후 생김)')

[
 {
  "실험": "A 지연(구매 시점)",
  "검증 기간으로 고른 방법": "로지스틱 회귀",
  "그 방법의 시험 성능": "AUC 0.745"
 },
 {
  "실험": "B 지연 what-if",
  "검증 기간으로 고른 방법": "HGB 단조 제약 (what-if가 상식에 맞게)",
  "그 방법의 시험 성능": "AUC 0.788"
 },
 {
  "실험": "C 도착일",
  "검증 기간으로 고른 방법": "선형 회귀",
  "그 방법의 시험 성능": "MAE 4.47일"
 },
 {
  "실험": "D 1~2점(배송 후)",
  "검증 기간으로 고른 방법": "HGB (부스팅)",
  "그 방법의 시험 성능": "AUC 0.699"
 },
 {
  "실험": "E 수요",
  "검증 기간으로 고른 방법": "지수가중평균(8주)",
  "그 방법의 시험 성능": "WAPE 0.747"
 },
 {
  "실험": "F 리뷰 글",
  "검증 기간으로 고른 방법": "TF-IDF + 로지스틱",
  "그 방법의 시험 성능": "AUC 0.953"
 }
]
